# Подготовка данных

Цель этого ноутбука — сформировать датасет для прогнозирования задержек доставки.

Прогноз выполняется в момент оформления заказа. Поэтому в модель могут входить только данные, доступные к этому моменту: дата покупки, обещанный срок доставки, характеристики покупателя, продавцов, товаров и состава заказа.

Фактические даты последующих логистических этапов не используются как признаки раннего прогноза.


In [1]:
import pandas as pd
import numpy as np

In [2]:
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv', parse_dates=['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
                                                                         'order_delivered_customer_date', 'order_estimated_delivery_date'])
print(orders.shape)

(99441, 8)


## Формирование обучающей выборки и целевой переменной

Для обучения оставлены только заказы со статусом `delivered`, поскольку только для них известен фактический результат доставки. Заказы с пропуском фактической даты доставки исключены.

Целевая переменная `is_delayed` показывает, был ли заказ доставлен позже обещанной календарной даты. Доставка в любой момент обещанного дня считается выполненной вовремя.

In [3]:
delivered_orders = orders[orders['order_status'] == 'delivered'].dropna(subset='order_delivered_customer_date')
print(delivered_orders.shape)
print(delivered_orders.isna().sum())

(96470, 8)
order_id                          0
customer_id                       0
order_status                      0
order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      1
order_delivered_customer_date     0
order_estimated_delivery_date     0
dtype: int64


In [4]:
delivered_orders['is_delayed'] = delivered_orders['order_delivered_customer_date'].dt.date > delivered_orders['order_estimated_delivery_date'].dt.date

In [5]:
delivered_orders['is_delayed'].value_counts()

is_delayed
False    89936
True      6534
Name: count, dtype: int64

In [6]:
delivered_orders['is_delayed'].value_counts(normalize=True)

is_delayed
False    0.932269
True     0.067731
Name: proportion, dtype: float64

После отбора в выборке осталось 96 470 заказов: 89 936 были доставлены вовремя и 6 534 — с задержкой. Доля задержанных заказов составляет около 6,8%.

Целевая переменная несбалансирована, поэтому при оценке моделей нельзя ориентироваться только на accuracy. Позже потребуются метрики, учитывающие качество определения редкого положительного класса.

## Выбор признаков и предотвращение утечки данных

Прогноз строится в момент оформления заказа. Поэтому в модель включаются только признаки, которые могут быть известны к этому моменту: дата покупки, обещанный срок доставки, характеристики покупателя, состав заказа, характеристики товаров и продавцов.

Фактические даты передачи перевозчику и доставки покупателю становятся известны позже и не используются как признаки. `order_delivered_customer_date` применяется только для формирования целевой переменной.

Идентификаторы используются как технические ключи для объединения таблиц, но не передаются модели. Данные отзывов также исключаются, поскольку появляются после доставки заказа.


## Подготовка географических данных

Таблица `geolocation` содержит несколько координат для одного ZIP-префикса. Среди координат встречаются отдельные значения, расположенные за пределами Бразилии. Такие значения считаются ошибочными и исключаются перед дальнейшей обработкой.

Для проверки используются приблизительные географические границы Бразилии. После удаления явно некорректных координат данные агрегируются: для каждого ZIP-префикса рассчитываются медианные широта и долгота.

В результате одному ZIP-префиксу соответствует одна строка. Это позволяет присоединить координаты к покупателям и продавцам без размножения записей, а использование медианы дополнительно уменьшает влияние отдельных неточных координат.


In [7]:
geolocation = pd.read_csv(
    '../data/raw/olist_geolocation_dataset.csv'
)

valid_coordinates = (
    geolocation['geolocation_lat'].between(-34, 6)
    & geolocation['geolocation_lng'].between(-74, -28)
)

geolocation_clean = (
    geolocation
    .loc[valid_coordinates]
    .copy()
)

print(
    'Удалено координат за пределами Бразилии:',
    len(geolocation) - len(geolocation_clean)
)

geolocation_by_zip = (
    geolocation_clean
    .groupby(
        'geolocation_zip_code_prefix',
        as_index=False
    )[
        ['geolocation_lat', 'geolocation_lng']
    ]
    .median()
)

print('Размер после агрегации:', geolocation_by_zip.shape)

print(
    'ZIP-префикс уникален:',
    geolocation_by_zip[
        'geolocation_zip_code_prefix'
    ].is_unique
)

Удалено координат за пределами Бразилии: 31
Размер после агрегации: (19011, 3)
ZIP-префикс уникален: True


### Присоединение координат покупателей

В таблице `customers` отсутствуют координаты покупателей, но присутствует ZIP-префикс. По нему к каждому покупателю присоединяются медианные широта и долгота из подготовленной таблицы `geolocation_by_zip`.

Используется соединение типа `many-to-one`: несколько покупателей могут иметь одинаковый ZIP-префикс, но в подготовленной географической таблице каждому ZIP соответствует только одна строка.


In [8]:
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')

customers_geo = customers.merge(
    geolocation_by_zip,
    how='left',
    left_on='customer_zip_code_prefix',
    right_on='geolocation_zip_code_prefix',
    validate='many_to_one'
)

customers_geo = (
    customers_geo
    .drop(columns='geolocation_zip_code_prefix')
    .rename(columns={
        'geolocation_lat': 'customer_lat',
        'geolocation_lng': 'customer_lng'
    })
)

In [9]:
orders_customers = delivered_orders.merge(
    customers_geo,
    how='left',
    on='customer_id',
    validate='one_to_one'
)

После присоединения покупателей каждому доставленному заказу по `customer_id` были добавлены характеристики покупателя и подготовленные координаты.

Соединение выполняется с проверкой связи `one-to-one`, поэтому одному заказу соответствует не более одной записи о покупателе. Для части покупателей корректные координаты определить не удалось; эти значения сохраняются как пропуски и будут обработаны позднее.


## Формирование признаков состава заказа

Таблица `order_items` содержит отдельную строку для каждой позиции заказа. Чтобы получить характеристики товаров и продавцов, к ней присоединяются таблицы `products` и `sellers` по ключам `product_id` и `seller_id`.

К продавцам предварительно добавляются координаты по ZIP-префиксу. Они будут использованы для расчёта расстояния между продавцом и покупателем до агрегации данных до уровня заказа.

Также для каждой позиции рассчитывается объём товара. Затем расширенная таблица будет агрегирована по `order_id`, чтобы одна строка соответствовала одному заказу.


In [10]:
order_items = pd.read_csv(
    '../data/raw/olist_order_items_dataset.csv',
    parse_dates=['shipping_limit_date']
)

In [11]:
products = pd.read_csv(
    '../data/raw/olist_products_dataset.csv'
)

# Товар не может весить 0 граммов, поэтому считаем такой вес пропуском.
# В остальных габаритах нулевых значений обнаружено не было.
products['product_weight_g'] = (
    products['product_weight_g']
    .replace(0, np.nan)
)

# Исправляем опечатки в названиях столбцов.
products = products.rename(columns={
    'product_name_lenght': 'product_name_length',
    'product_description_lenght': 'product_description_length'
})

items_products = order_items.merge(
    products,
    how='left',
    on='product_id',
    validate='many_to_one'
)


sellers = pd.read_csv(
    '../data/raw/olist_sellers_dataset.csv'
)

sellers_geo = sellers.merge(
    geolocation_by_zip,
    how='left',
    left_on='seller_zip_code_prefix',
    right_on='geolocation_zip_code_prefix',
    validate='many_to_one'
)

sellers_geo = (
    sellers_geo
    .drop(columns='geolocation_zip_code_prefix')
    .rename(columns={
        'geolocation_lat': 'seller_lat',
        'geolocation_lng': 'seller_lng'
    })
)

In [12]:
items_products_sellers = items_products.merge(
    sellers_geo,
    how='left',
    on='seller_id',
    validate='many_to_one'
)

In [13]:
#вместо отдельных длины, высоты и ширины, создадим общий параметр объема
items_products_sellers['product_volume_cm3'] = (
    items_products_sellers['product_length_cm']
    * items_products_sellers['product_height_cm']
    * items_products_sellers['product_width_cm']
)

### Расчёт расстояния между покупателем и продавцом

Для каждой позиции заказа к координатам продавца добавляются координаты покупателя по `order_id`. После этого рассчитывается приблизительное расстояние между двумя точками по формуле гаверсинуса.

Формула учитывает сферическую форму Земли и возвращает расстояние по её поверхности в километрах. Полученное значение не соответствует точной длине автомобильного маршрута, но позволяет приблизительно охарактеризовать удалённость продавца от покупателя.

Если заказ включает позиции от нескольких продавцов, при последующей агрегации будет использовано максимальное расстояние. Оно характеризует наиболее удалённую отправку в составе заказа.


In [14]:
items_products_sellers = items_products_sellers.merge(
    orders_customers[
        ['order_id', 'customer_lat', 'customer_lng']
    ],
    how='left',
    on='order_id',
    validate='many_to_one'
)

In [15]:
customer_lat_rad = np.radians(
    items_products_sellers['customer_lat']
)
customer_lng_rad = np.radians(
    items_products_sellers['customer_lng']
)
seller_lat_rad = np.radians(
    items_products_sellers['seller_lat']
)
seller_lng_rad = np.radians(
    items_products_sellers['seller_lng']
)

lat_difference = seller_lat_rad - customer_lat_rad
lng_difference = seller_lng_rad - customer_lng_rad

a = (
    np.sin(lat_difference / 2) ** 2
    + np.cos(customer_lat_rad)
    * np.cos(seller_lat_rad)
    * np.sin(lng_difference / 2) ** 2
)

a = np.clip(a, 0, 1)

angular_distance = 2 * np.arctan2(
    np.sqrt(a),
    np.sqrt(1 - a)
)

earth_radius_km = 6371

items_products_sellers['delivery_distance_km'] = (
    earth_radius_km * angular_distance
)

In [16]:
items_product_sellers_by_order = (
    items_products_sellers
    .groupby('order_id')
    .agg(
        items=('order_item_id', 'size'),
        products=('product_id', 'nunique'),
        sellers=('seller_id', 'nunique'),
        total_price=('price', 'sum'),
        total_freight_value=('freight_value', 'sum'),
        categories=('product_category_name', 'nunique'),
        avg_name_len=('product_name_length', 'mean'),
        avg_desc_len=('product_description_length', 'mean'),
        avg_photos_per_item=('product_photos_qty', 'mean'),
        total_weight=(
            'product_weight_g',
            lambda x: x.sum(min_count=1)
        ),
        total_volume=(
            'product_volume_cm3',
            lambda x: x.sum(min_count=1)
        ),
        max_volume=('product_volume_cm3', 'max'),
        seller_states=('seller_state', 'nunique'),
        max_delivery_distance_km=(
            'delivery_distance_km',
            'max'
        ),
        main_seller_state=(
            'seller_state',
            lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
        ),
        main_product_category=(
            'product_category_name',
            lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
        )
    )
    .reset_index()
)

Для каждого заказа рассчитаны количество позиций, уникальных товаров, продавцов, категорий и штатов продавцов; суммарная стоимость товаров и доставки; средние характеристики карточек товаров; общий вес и объём заказа, а также максимальный объём одной позиции.

Дополнительно рассчитаны максимальное расстояние до продавца, основной штат продавца и основная категория товара. Основными считаются наиболее часто встречающиеся значения среди позиций заказа. Если несколько значений встречаются одинаково часто, выбирается первое значение из результата `mode()`.

Для заказа с одним продавцом максимальное расстояние равно расстоянию до этого продавца. Для заказа с несколькими продавцами сохраняется расстояние до наиболее удалённого продавца.

При суммировании веса и объёма полностью отсутствующие значения сохраняются как пропуски, а не превращаются в физически невозможные нули. Пропуски в характеристиках товаров, продавцов и расстоянии будут обработаны позднее внутри модельного пайплайна.


## Объединение признаков

Агрегированные характеристики состава заказа присоединяются к таблице заказов и покупателей по `order_id`. Левое соединение сохраняет все отобранные доставленные заказы, а проверка связи `one-to-one` предотвращает размножение строк.


In [17]:
dataset = orders_customers.merge(
    items_product_sellers_by_order,
    how='left',
    on='order_id',
    validate='one_to_one'
)

После объединения сравниваются штат покупателя и основной штат продавца. Признак `same_state_delivery` принимает значение `1`, если штаты совпадают, и `0`, если различаются.

Если основной штат продавца неизвестен, значение признака сохраняется как пропуск. Это позволяет позднее обработать его внутри модельного пайплайна вместе с остальными пропусками.


In [18]:
dataset['same_state_delivery'] = (
    dataset['customer_state']
    .eq(dataset['main_seller_state'])
    .where(dataset['main_seller_state'].notna())
    .astype(float)
)

In [19]:
dataset.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_delayed,customer_unique_id,...,avg_desc_len,avg_photos_per_item,total_weight,total_volume,max_volume,seller_states,max_delivery_distance_km,main_seller_state,main_product_category,same_state_delivery
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,False,7c396fd4830fd04220f754e42b4e5bff,...,268.0,4.0,500.0,1976.0,1976.0,1,18.681711,SP,utilidades_domesticas,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,False,af07308b275d755c9edb36a90c618231,...,178.0,1.0,400.0,4693.0,4693.0,1,861.035367,SP,perfumaria,0.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,False,3a653a41f6f9fc3d2a113cf8398680e8,...,232.0,1.0,420.0,9576.0,9576.0,1,514.547140,SP,automotivo,0.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,False,7c142cf63193a1473d2e66489a9ae977,...,468.0,3.0,450.0,6000.0,6000.0,1,1821.802656,MG,pet_shop,0.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,False,72632f0f9dd73dfee390c9b22eb56dd6,...,316.0,4.0,250.0,11475.0,11475.0,1,29.593095,SP,papelaria,1.0


In [20]:
dataset.shape[0] == orders_customers.shape[0]

True

In [21]:
print(
    'Пропусков в расстоянии:',
    dataset['max_delivery_distance_km'].isna().sum()
)

dataset['max_delivery_distance_km'].describe()

Пропусков в расстоянии: 477


count    95993.000000
mean       601.778506
std        592.645211
min          0.000000
25%        188.634560
50%        435.528467
75%        800.185076
max       3398.485605
Name: max_delivery_distance_km, dtype: float64

In [22]:
dataset['order_purchase_month'] = dataset['order_purchase_timestamp'].dt.month
dataset['order_purchase_dayofweek'] = dataset['order_purchase_timestamp'].dt.dayofweek
dataset['order_purchase_hour'] = dataset['order_purchase_timestamp'].dt.hour
dataset['estimated_delivery_days'] = (dataset['order_estimated_delivery_date'] - dataset['order_purchase_timestamp']) / pd.Timedelta(days=1)

In [23]:
dataset[['order_purchase_month', 'order_purchase_dayofweek', 'order_purchase_hour', 'estimated_delivery_days']].describe()

,order_purchase_month,order_purchase_dayofweek,order_purchase_hour,estimated_delivery_days
count,96470.000000,96470.000000,96470.000000,96470.000000
mean,6.031046,2.756494,14.773028,23.736343
std,3.228479,1.967041,5.328347,8.761052
min,1.000000,0.000000,0.000000,2.008009
25%,3.000000,1.000000,11.000000,18.329905
50%,6.000000,3.000000,15.000000,23.230880
75%,8.000000,4.000000,19.000000,28.407795
max,12.000000,6.000000,23.000000,155.135463


## Итоги подготовки данных

Сформирован датасет из 96 470 доставленных заказов, где одна строка соответствует одному заказу. Для каждого заказа создана целевая переменная `is_delayed`: доля задержанных заказов составляет около 6,8%, поэтому при дальнейшем моделировании необходимо учитывать дисбаланс классов.

К данным о заказах добавлены характеристики покупателей, состава заказа, товаров и продавцов. Также созданы временные признаки: месяц, день недели и час оформления заказа, а также обещанный срок доставки в днях.

Для покупателей и продавцов подготовлены координаты на уровне ZIP-префиксов. Координаты, расположенные за пределами Бразилии, исключены как ошибочные. На основе оставшихся значений рассчитано приблизительное расстояние между покупателем и каждым продавцом, а для каждого заказа сохранено максимальное расстояние.

Дополнительно созданы признаки основного штата продавца, основной категории товара и совпадения штатов покупателя и продавца. Для 477 заказов расстояние рассчитать не удалось из-за отсутствия корректных координат. Эти и другие пропуски сохранены для последующей обработки внутри модельного пайплайна после разделения данных.

Идентификаторы и фактические логистические даты сохранены в датасете как технические столбцы, но не будут передаваться модели. Подготовленные данные позволяют сравнить исходный набор признаков с расширенным набором, включающим расстояние доставки, основной штат продавца, основную категорию товара и совпадение штатов.


In [24]:
dataset.to_csv(
    '../data/processed/dataset.csv',
    index=False
)